# Top 10 Backtest Configurations Comparison

Compares IchiV2 and IchiV3 WhaleCap strategy configurations across different pair universes and slot settings.

**Experiments:**
- exp017: IchiV2 WC pair universe sweep (Binance volume pairlist) — Top30, Top40, Top50
- exp007: IchiV3 WC pair universe sweep (Binance volume pairlist) — Top30, Top40, Top50
- exp020: V2 vs V3 final showdown (static 107 pairs) — V2 Best, V2 6slot, V3 Best, V3 NoPool (live config)

**Period:** 2021-07-18 to 2026-03-12 | **Starting capital:** $100,000

In [1]:
import zipfile
import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_dark"

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

INITIAL_CAPITAL = 100_000
BT_START = pd.Timestamp("2021-07-18", tz="UTC")
BT_END = pd.Timestamp("2026-03-12", tz="UTC")
YEARS = (BT_END - BT_START).days / 365.25
print(f"Backtest span: {YEARS:.2f} years")

Backtest span: 4.65 years


---
## 1. Data Loading

In [2]:
CONFIGS = [
    # (label, zip_path, pairlist_type)
    ("V2 WC Top30",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv2-gmx/017-pair-universe-sweep/results/arm_top30.zip",
     "Binance Volume"),
    ("V2 WC Top40",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv2-gmx/017-pair-universe-sweep/results/arm_top40.zip",
     "Binance Volume"),
    ("V2 WC Top50",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv2-gmx/017-pair-universe-sweep/results/arm_top50.zip",
     "Binance Volume"),
    ("V3 WC Top30",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv3-gmx/007-pair-universe-sweep/results/arm_a_top30.zip",
     "Binance Volume"),
    ("V3 WC Top40",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv3-gmx/007-pair-universe-sweep/results/arm_b_top40.zip",
     "Binance Volume"),
    ("V3 WC Top50",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv3-gmx/007-pair-universe-sweep/results/arm_c_top50.zip",
     "Binance Volume"),
    ("V2 WC Static",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv2-gmx/020-v2wc-vs-v3wc-final/results/arm_a_v2wc_best.zip",
     "Static 107"),
    ("V2 WC 6slot",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv2-gmx/020-v2wc-vs-v3wc-final/results/arm_c_v2wc_6slots.zip",
     "Static 107"),
    ("V3 WC Static",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv2-gmx/020-v2wc-vs-v3wc-final/results/arm_d_v3wc_best.zip",
     "Static 107"),
    ("V3 WC NoPool",
     "/home/ubuntu/dev/gmx-ccxt-freqtrade/experiments/ichiv2-gmx/020-v2wc-vs-v3wc-final/results/arm_f_v3wc_no_pool.zip",
     "Static 107"),
]

def load_backtest_zip(zip_path):
    """Load backtest JSON from a ZIP file. Returns (strategy_data, strategy_name)."""
    with zipfile.ZipFile(zip_path) as zf:
        json_files = [f for f in zf.namelist() if f.endswith(".json") and "_config" not in f
                      and not f.endswith(".py") and "market_change" not in f
                      and f.count("_") <= 6]  # main result file
        # Pick the shortest-named json (the main result)
        json_files = sorted(json_files, key=len)
        # Filter to only the main backtest result file
        candidates = [f for f in zf.namelist()
                      if f.startswith("backtest-result-") and f.endswith(".json")
                      and "_config" not in f and "_market" not in f
                      and not any(f.endswith(f"_{suf}.json") for suf in ["config"])]
        # The main result file has the pattern backtest-result-DATETIME.json (no extra suffix before .json)
        main_files = [f for f in candidates if f.count("_") <= 4]
        if not main_files:
            # fallback: file with 'strategy' key
            for f in candidates:
                data = json.loads(zf.read(f))
                if "strategy" in data:
                    strat_name = list(data["strategy"].keys())[0]
                    return data["strategy"][strat_name], strat_name
        fname = main_files[0] if main_files else candidates[0]
        data = json.loads(zf.read(fname))
        strat_name = list(data["strategy"].keys())[0]
        return data["strategy"][strat_name], strat_name

# Load all configs
results = {}
for label, zpath, pl_type in CONFIGS:
    strat_data, strat_name = load_backtest_zip(zpath)
    results[label] = {
        "data": strat_data,
        "strat_name": strat_name,
        "pairlist": pl_type,
    }
    n_trades = strat_data["total_trades"]
    pnl = strat_data["profit_total_abs"]
    print(f"  {label:20s} | {strat_name:40s} | {n_trades:>5} trades | PnL ${pnl:>12,.2f} | {pl_type}")

print(f"\nLoaded {len(results)} configurations.")

  V2 WC Top30          | IchiV2_LS_Static_WhaleCap                |  1380 trades | PnL $   82,530.37 | Binance Volume
  V2 WC Top40          | IchiV2_LS_Static_WhaleCap                |  1666 trades | PnL $  143,410.64 | Binance Volume


  V2 WC Top50          | IchiV2_LS_Static_WhaleCap                |  1858 trades | PnL $  166,621.37 | Binance Volume
  V3 WC Top30          | IchiV3_LS_Static_WhaleCap                |  1214 trades | PnL $  104,208.09 | Binance Volume
  V3 WC Top40          | IchiV3_LS_Static_WhaleCap                |  1410 trades | PnL $  135,573.11 | Binance Volume


  V3 WC Top50          | IchiV3_LS_Static_WhaleCap                |  1497 trades | PnL $  175,980.40 | Binance Volume
  V2 WC Static         | IchiV2_LS_Static_WhaleCap                |  2049 trades | PnL $  142,314.61 | Static 107


  V2 WC 6slot          | IchiV2_LS_Static_WhaleCap                |  1651 trades | PnL $  191,046.81 | Static 107
  V3 WC Static         | IchiV3_LS_Static_WhaleCap                |  1626 trades | PnL $  149,790.37 | Static 107
  V3 WC NoPool         | IchiV3_LS_Static_WhaleCap                |  1647 trades | PnL $  176,394.87 | Static 107

Loaded 10 configurations.


In [3]:
def build_trades_df(strat_data, label):
    """Convert trades list to a DataFrame with proper types."""
    trades = strat_data["trades"]
    df = pd.DataFrame(trades)
    df["open_date"] = pd.to_datetime(df["open_date"], utc=True)
    df["close_date"] = pd.to_datetime(df["close_date"], utc=True)
    df["config"] = label
    return df

def build_daily_pnl(trades_df):
    """Build daily P&L series from trade close dates."""
    daily = trades_df.groupby(trades_df["close_date"].dt.date)["profit_abs"].sum()
    daily.index = pd.to_datetime(daily.index, utc=True)
    # Reindex to full date range
    full_range = pd.date_range(BT_START, BT_END, freq="D", tz="UTC")
    daily = daily.reindex(full_range, fill_value=0.0)
    return daily

def build_equity_curve(daily_pnl):
    """Cumulative equity starting from INITIAL_CAPITAL."""
    return INITIAL_CAPITAL + daily_pnl.cumsum()

def calc_drawdown_series(equity):
    """Return drawdown series (negative values) and peak series."""
    peak = equity.cummax()
    dd = (equity - peak) / peak  # fractional drawdown
    return dd, peak

# Build all DataFrames
all_trades = {}
daily_pnls = {}
equity_curves = {}
dd_series = {}

for label in results:
    tdf = build_trades_df(results[label]["data"], label)
    all_trades[label] = tdf
    dpnl = build_daily_pnl(tdf)
    daily_pnls[label] = dpnl
    eq = build_equity_curve(dpnl)
    equity_curves[label] = eq
    dd, _ = calc_drawdown_series(eq)
    dd_series[label] = dd

print("Built equity curves and drawdown series for all configs.")

Built equity curves and drawdown series for all configs.


---
## 2. Summary Metrics Table

In [4]:
def calc_metrics(label):
    """Calculate all summary metrics for a config."""
    sd = results[label]["data"]
    tdf = all_trades[label]
    dpnl = daily_pnls[label]
    eq = equity_curves[label]
    dd, peak = calc_drawdown_series(eq)

    total_trades = sd["total_trades"]
    winrate = sd["winrate"] * 100
    total_profit_abs = sd["profit_total_abs"]
    total_profit_pct = sd["profit_total"] * 100

    # CAGR
    final_val = eq.iloc[-1]
    cagr = ((final_val / INITIAL_CAPITAL) ** (1 / YEARS) - 1) * 100

    # Sharpe (annualised, from daily returns)
    daily_ret = dpnl / eq.shift(1).fillna(INITIAL_CAPITAL)
    sharpe = (daily_ret.mean() / daily_ret.std()) * np.sqrt(365) if daily_ret.std() > 0 else 0

    # Sortino (annualised, downside deviation)
    downside = daily_ret[daily_ret < 0]
    downside_std = np.sqrt((downside ** 2).mean()) if len(downside) > 0 else 1e-9
    sortino = (daily_ret.mean() / downside_std) * np.sqrt(365)

    # Max drawdown
    max_dd = dd.min() * 100  # negative percentage

    # Calmar
    calmar = cagr / abs(max_dd) if max_dd != 0 else 0

    # Profit factor
    profit_factor = sd.get("profit_factor", 0)

    # Expectancy ratio
    expectancy_ratio = sd.get("expectancy_ratio", 0)

    # Max drawdown duration (days)
    # Find longest period where equity was below its running peak
    in_dd = dd < 0
    if in_dd.any():
        # Group consecutive True values
        groups = (~in_dd).cumsum()
        dd_groups = in_dd.groupby(groups)
        max_dd_duration = max(g.sum() for _, g in dd_groups)
    else:
        max_dd_duration = 0

    # Average trade duration (minutes -> hours)
    avg_dur_min = tdf["trade_duration"].mean()
    avg_dur_hours = avg_dur_min / 60

    return {
        "Config": label,
        "Pairlist": results[label]["pairlist"],
        "Total Trades": total_trades,
        "Win Rate %": winrate,
        "Total Profit $": total_profit_abs,
        "Total Profit %": total_profit_pct,
        "CAGR %": cagr,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max DD %": max_dd,
        "Calmar": calmar,
        "Profit Factor": profit_factor,
        "Expectancy Ratio": expectancy_ratio,
        "Max DD Duration (d)": max_dd_duration,
        "Avg Trade Dur (h)": avg_dur_hours,
    }

metrics_list = [calc_metrics(label) for label in results]
metrics_df = pd.DataFrame(metrics_list)
metrics_df = metrics_df.sort_values("Calmar", ascending=False).reset_index(drop=True)

# Format for display
fmt_df = metrics_df.copy()
fmt_df["Total Profit $"] = fmt_df["Total Profit $"].apply(lambda x: f"${x:,.0f}")
fmt_df["Total Profit %"] = fmt_df["Total Profit %"].apply(lambda x: f"{x:.1f}%")
fmt_df["Win Rate %"] = fmt_df["Win Rate %"].apply(lambda x: f"{x:.1f}%")
fmt_df["CAGR %"] = fmt_df["CAGR %"].apply(lambda x: f"{x:.1f}%")
fmt_df["Max DD %"] = fmt_df["Max DD %"].apply(lambda x: f"{x:.1f}%")
fmt_df["Sharpe"] = fmt_df["Sharpe"].apply(lambda x: f"{x:.2f}")
fmt_df["Sortino"] = fmt_df["Sortino"].apply(lambda x: f"{x:.2f}")
fmt_df["Calmar"] = fmt_df["Calmar"].apply(lambda x: f"{x:.2f}")
fmt_df["Profit Factor"] = fmt_df["Profit Factor"].apply(lambda x: f"{x:.2f}")
fmt_df["Expectancy Ratio"] = fmt_df["Expectancy Ratio"].apply(lambda x: f"{x:.3f}")
fmt_df["Max DD Duration (d)"] = fmt_df["Max DD Duration (d)"].astype(int)
fmt_df["Avg Trade Dur (h)"] = fmt_df["Avg Trade Dur (h)"].apply(lambda x: f"{x:.1f}")
fmt_df["Total Trades"] = fmt_df["Total Trades"].apply(lambda x: f"{x:,}")

display(fmt_df.style.set_caption("Summary Metrics — Sorted by Calmar Ratio (descending)"))

,Config,Pairlist,Total Trades,Win Rate %,Total Profit $,Total Profit %,CAGR %,Sharpe,Sortino,Max DD %,Calmar,Profit Factor,Expectancy Ratio,Max DD Duration (d),Avg Trade Dur (h)
0,V2 WC Top40,Binance Volume,"1,666",39.1%,"$143,411",143.4%,21.1%,1.44,1.75,-16.7%,1.26,1.36,0.218,515,640.2
1,V2 WC Top50,Binance Volume,"1,858",39.4%,"$166,621",166.6%,23.5%,1.42,1.80,-21.6%,1.09,1.38,0.230,519,515.6
2,V3 WC Top50,Binance Volume,"1,497",39.7%,"$175,980",176.0%,24.4%,1.14,1.28,-26.3%,0.93,1.30,0.182,502,408.4
3,V2 WC Top30,Binance Volume,"1,380",38.3%,"$82,530",82.5%,13.8%,1.18,1.34,-15.1%,0.92,1.25,0.157,543,1293.7
4,V2 WC Static,Static 107,"2,049",38.5%,"$142,315",142.3%,21.0%,1.16,1.56,-24.2%,0.87,1.31,0.189,512,48.5
5,V2 WC 6slot,Static 107,"1,651",37.1%,"$191,047",191.0%,25.8%,1.15,1.46,-30.2%,0.85,1.31,0.195,512,46.7
6,V3 WC Top40,Binance Volume,"1,410",39.6%,"$135,573",135.6%,20.2%,1.06,1.14,-24.0%,0.84,1.25,0.152,500,590.8
7,V3 WC NoPool,Static 107,"1,647",38.9%,"$176,395",176.4%,24.4%,1.08,1.29,-29.1%,0.84,1.28,0.171,489,48.1
8,V3 WC Static,Static 107,"1,626",38.9%,"$149,790",149.8%,21.8%,1.03,1.19,-28.6%,0.76,1.26,0.156,495,47.8
9,V3 WC Top30,Binance Volume,"1,214",38.3%,"$104,208",104.2%,16.6%,0.97,1.04,-22.4%,0.74,1.22,0.134,542,1272.5


---
## 3. Equity Curves

In [5]:
# Color scheme: V2 = blues, V3 = reds/oranges
COLOR_MAP = {
    "V2 WC Top30": "#1f77b4",
    "V2 WC Top40": "#2196F3",
    "V2 WC Top50": "#64B5F6",
    "V2 WC Static": "#0D47A1",
    "V2 WC 6slot": "#4FC3F7",
    "V3 WC Top30": "#e53935",
    "V3 WC Top40": "#FF7043",
    "V3 WC Top50": "#FFB74D",
    "V3 WC Static": "#D32F2F",
    "V3 WC NoPool": "#FFD600",  # Live config — bright yellow
}

LIVE_CONFIG = "V3 WC NoPool"

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.04,
    subplot_titles=("Equity Curves ($100k start)", "Drawdown from Peak")
)

for label in equity_curves:
    eq = equity_curves[label]
    dd = dd_series[label]
    color = COLOR_MAP.get(label, "gray")
    is_live = label == LIVE_CONFIG
    width = 3 if is_live else 1.5
    dash = "dash" if is_live else "solid"
    legend_name = f"{label} (LIVE)" if is_live else label

    fig.add_trace(
        go.Scatter(
            x=eq.index, y=eq.values,
            name=legend_name, line=dict(color=color, width=width, dash=dash),
            hovertemplate="%{x|%Y-%m-%d}<br>$%{y:,.0f}<extra>" + label + "</extra>",
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=dd.index, y=dd.values * 100,
            name=label, line=dict(color=color, width=width if is_live else 1, dash=dash),
            showlegend=False,
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.1f}%<extra>" + label + "</extra>",
        ),
        row=2, col=1,
    )

fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
fig.update_yaxes(title_text="Drawdown %", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_layout(
    height=800, width=1200,
    title="All Configs — Equity Curves & Drawdowns",
    legend=dict(font=dict(size=10)),
    hovermode="x unified",
)
fig.show()

---
## 4. Drawdown Comparison

In [6]:
fig_dd = go.Figure()

for label in dd_series:
    dd = dd_series[label]
    color = COLOR_MAP.get(label, "gray")
    is_live = label == LIVE_CONFIG
    width = 3 if is_live else 1.2
    dash = "dash" if is_live else "solid"

    fig_dd.add_trace(
        go.Scatter(
            x=dd.index, y=dd.values * 100,
            name=f"{label} (LIVE)" if is_live else label,
            line=dict(color=color, width=width, dash=dash),
            fill="tozeroy" if is_live else None,
            fillcolor=f"rgba(255,214,0,0.10)" if is_live else None,
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.1f}%<extra>" + label + "</extra>",
        )
    )

fig_dd.update_layout(
    height=500, width=1200,
    title="Underwater (Drawdown from Peak) — All Configs",
    yaxis_title="Drawdown %",
    xaxis_title="Date",
    hovermode="x unified",
)
fig_dd.show()

---
## 5. Rolling 90-Day Sharpe

In [7]:
ROLLING_WINDOW = 90

fig_rs = go.Figure()

for label in daily_pnls:
    dpnl = daily_pnls[label]
    eq = equity_curves[label]
    daily_ret = dpnl / eq.shift(1).fillna(INITIAL_CAPITAL)

    rolling_mean = daily_ret.rolling(ROLLING_WINDOW).mean()
    rolling_std = daily_ret.rolling(ROLLING_WINDOW).std()
    rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(365)

    color = COLOR_MAP.get(label, "gray")
    is_live = label == LIVE_CONFIG
    width = 3 if is_live else 1.2
    dash = "dash" if is_live else "solid"

    fig_rs.add_trace(
        go.Scatter(
            x=rolling_sharpe.index, y=rolling_sharpe.values,
            name=f"{label} (LIVE)" if is_live else label,
            line=dict(color=color, width=width, dash=dash),
            hovertemplate="%{x|%Y-%m-%d}<br>Sharpe: %{y:.2f}<extra>" + label + "</extra>",
        )
    )

fig_rs.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.5)
fig_rs.update_layout(
    height=500, width=1200,
    title=f"Rolling {ROLLING_WINDOW}-Day Annualised Sharpe Ratio",
    yaxis_title="Sharpe Ratio",
    xaxis_title="Date",
    hovermode="x unified",
)
fig_rs.show()

---
## 6. Monthly Returns Heatmaps (Top 3 by Calmar)

In [8]:
top3_calmar = metrics_df.head(3)["Config"].tolist()
print(f"Top 3 by Calmar: {top3_calmar}")

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for label in top3_calmar:
    dpnl = daily_pnls[label]
    # Monthly returns as % of starting equity for that month
    eq = equity_curves[label]

    # Group by year-month
    monthly_pnl = dpnl.groupby([dpnl.index.year, dpnl.index.month]).sum()
    monthly_eq_start = eq.groupby([eq.index.year, eq.index.month]).first()
    monthly_ret = (monthly_pnl / monthly_eq_start.shift(1).fillna(INITIAL_CAPITAL)) * 100

    # Pivot to year x month
    years = sorted(set(dpnl.index.year))
    months = list(range(1, 13))
    heatmap_data = pd.DataFrame(index=years, columns=months, dtype=float)
    for (y, m), val in monthly_ret.items():
        heatmap_data.loc[y, m] = val

    fig_hm = go.Figure(
        data=go.Heatmap(
            z=heatmap_data.values,
            x=month_names,
            y=[str(y) for y in heatmap_data.index],
            colorscale="RdYlGn",
            zmid=0,
            text=np.where(
                pd.isna(heatmap_data.values),
                "",
                np.vectorize(lambda x: f"{x:.1f}%")(np.nan_to_num(heatmap_data.values))
            ),
            texttemplate="%{text}",
            hovertemplate="%{y} %{x}<br>Return: %{z:.1f}%<extra></extra>",
        )
    )
    fig_hm.update_layout(
        title=f"Monthly Returns % — {label}",
        height=350, width=900,
        yaxis=dict(autorange="reversed"),
    )
    fig_hm.show()

Top 3 by Calmar: ['V2 WC Top40', 'V2 WC Top50', 'V3 WC Top50']


---
## 7. Trade Distribution Analysis

In [9]:
top5_calmar = metrics_df.head(5)["Config"].tolist()
print(f"Top 5 by Calmar: {top5_calmar}")

# Combine trades for top 5
top5_trades = pd.concat([all_trades[l] for l in top5_calmar], ignore_index=True)

# Profit ratio violin
fig_violin = go.Figure()
for label in top5_calmar:
    tdf = all_trades[label]
    color = COLOR_MAP.get(label, "gray")
    fig_violin.add_trace(
        go.Violin(
            y=tdf["profit_ratio"] * 100,
            name=label,
            line_color=color,
            meanline_visible=True,
            box_visible=True,
        )
    )
fig_violin.update_layout(
    title="Per-Trade Profit % Distribution (Top 5 Configs)",
    yaxis_title="Profit %",
    height=500, width=1000,
    showlegend=False,
)
fig_violin.show()

Top 5 by Calmar: ['V2 WC Top40', 'V2 WC Top50', 'V3 WC Top50', 'V2 WC Top30', 'V2 WC Static']


In [10]:
# Stake amount distribution
fig_stake = go.Figure()
for label in top5_calmar:
    tdf = all_trades[label]
    color = COLOR_MAP.get(label, "gray")
    fig_stake.add_trace(
        go.Box(
            y=tdf["stake_amount"],
            name=label,
            marker_color=color,
        )
    )
fig_stake.update_layout(
    title="Stake Amount Distribution (Top 5 Configs)",
    yaxis_title="Stake Amount ($)",
    height=450, width=1000,
    showlegend=False,
)
fig_stake.show()

In [11]:
# Trade duration distribution (in hours)
fig_dur = go.Figure()
for label in top5_calmar:
    tdf = all_trades[label]
    color = COLOR_MAP.get(label, "gray")
    dur_hours = tdf["trade_duration"] / 60
    fig_dur.add_trace(
        go.Box(
            y=dur_hours,
            name=label,
            marker_color=color,
        )
    )
fig_dur.update_layout(
    title="Trade Duration Distribution — Hours (Top 5 Configs)",
    yaxis_title="Duration (hours)",
    height=450, width=1000,
    showlegend=False,
)
fig_dur.show()

---
## 8. Per-Pair Breakdown (Top 3 Configs)

In [12]:
for label in top3_calmar:
    tdf = all_trades[label]
    pair_profit = tdf.groupby("pair")["profit_abs"].sum().sort_values(ascending=False)
    top10 = pair_profit.head(10)

    color = COLOR_MAP.get(label, "gray")
    fig_pair = go.Figure(
        go.Bar(
            x=top10.values,
            y=top10.index,
            orientation="h",
            marker_color=color,
            text=[f"${v:,.0f}" for v in top10.values],
            textposition="outside",
        )
    )
    fig_pair.update_layout(
        title=f"Top 10 Pairs by Profit — {label}",
        xaxis_title="Total Profit ($)",
        height=400, width=900,
        yaxis=dict(autorange="reversed"),
    )
    fig_pair.show()

---
## 9. Head-to-Head: V2 WC Top40 vs V3 WC NoPool

In [13]:
H2H_A = "V2 WC Top40"
H2H_B = "V3 WC NoPool"

# Side-by-side equity curves
fig_h2h = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    vertical_spacing=0.05,
    subplot_titles=("Equity Curves", "Drawdown from Peak"),
)

for label, color, dash in [(H2H_A, "#2196F3", "solid"), (H2H_B, "#FFD600", "dash")]:
    eq = equity_curves[label]
    dd = dd_series[label]
    fig_h2h.add_trace(
        go.Scatter(
            x=eq.index, y=eq.values, name=label,
            line=dict(color=color, width=2.5, dash=dash),
            hovertemplate="%{x|%Y-%m-%d}<br>$%{y:,.0f}<extra>" + label + "</extra>",
        ), row=1, col=1,
    )
    fig_h2h.add_trace(
        go.Scatter(
            x=dd.index, y=dd.values * 100, name=label,
            line=dict(color=color, width=2, dash=dash),
            showlegend=False,
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.1f}%<extra>" + label + "</extra>",
        ), row=2, col=1,
    )

fig_h2h.update_yaxes(title_text="Portfolio ($)", row=1, col=1)
fig_h2h.update_yaxes(title_text="DD %", row=2, col=1)
fig_h2h.update_layout(
    height=650, width=1100,
    title=f"Head-to-Head: {H2H_A} vs {H2H_B} (Live)",
    hovermode="x unified",
)
fig_h2h.show()

In [14]:
# Monthly return comparison
h2h_monthly = {}
for label in [H2H_A, H2H_B]:
    dpnl = daily_pnls[label]
    eq = equity_curves[label]
    monthly_pnl = dpnl.groupby(pd.Grouper(freq="MS")).sum()
    monthly_eq_start = eq.groupby(pd.Grouper(freq="MS")).first().shift(1).fillna(INITIAL_CAPITAL)
    h2h_monthly[label] = (monthly_pnl / monthly_eq_start) * 100

fig_monthly = go.Figure()
for label, color in [(H2H_A, "#2196F3"), (H2H_B, "#FFD600")]:
    mr = h2h_monthly[label]
    fig_monthly.add_trace(
        go.Bar(
            x=mr.index, y=mr.values, name=label,
            marker_color=color, opacity=0.7,
        )
    )

fig_monthly.update_layout(
    title=f"Monthly Returns % — {H2H_A} vs {H2H_B}",
    yaxis_title="Monthly Return %",
    xaxis_title="Date",
    barmode="group",
    height=450, width=1200,
)
fig_monthly.show()

In [15]:
# Statistical summary side by side
h2h_metrics = metrics_df[metrics_df["Config"].isin([H2H_A, H2H_B])].set_index("Config").T
display(h2h_metrics.style.set_caption(f"Head-to-Head Summary: {H2H_A} vs {H2H_B}"))

Config,V2 WC Top40,V3 WC NoPool
Pairlist,Binance Volume,Static 107
Total Trades,1666,1647
Win Rate %,39.135654,38.858531
Total Profit $,143410.642822,176394.866629
Total Profit %,143.410643,176.394867
CAGR %,21.088796,24.444512
Sharpe,1.443556,1.078517
Sortino,1.746986,1.287519
Max DD %,-16.694261,-29.082413
Calmar,1.263236,0.840526


---
## 10. Key Findings

### Risk-Adjusted Returns (Calmar / Sharpe / Sortino)
- The summary table above (sorted by Calmar) reveals which config delivers the best return per unit of drawdown risk.
- Configs using **Binance volume pairlists** (TopN) focus capital on higher-liquidity pairs, which can improve risk-adjusted metrics.
- The **static 107-pair** configs cast a wider net and may capture more trades but can also encounter lower-quality setups.

### Raw Returns
- Check the "Total Profit $" and "CAGR %" columns for the highest raw returns — a wider pair universe often generates more total profit.

### Current Live Config: V3 WC NoPool
- **V3 WC NoPool** is the current production configuration (highlighted with dashed yellow lines in all charts).
- This config runs IchiV3 WhaleCap on the static 107-pair universe **without** the liquidity pool filter.
- Its equity curve, drawdown profile, and rolling Sharpe should be compared against the TopN alternatives.

### Binance Volume Pairlist — Now Available for Live
- All "TopN" configs (exp017, exp007) relied on Binance 24h volume data for pair selection.
- This capability is now available in live trading via **HistoricalVolumePairList**, making these configs viable production candidates.
- If a TopN config shows superior risk-adjusted returns, it is a strong candidate to replace the current live setup.

### V2 vs V3
- The head-to-head section (Section 9) compares the top V2 config against the live V3 NoPool config.
- Key differences in monthly return patterns and drawdown behaviour inform whether a strategy version change is warranted.

### Recommendations
- **Best risk-adjusted**: See rank #1 in the Calmar-sorted table above.
- **Best raw returns**: See the config with highest CAGR%.
- **Next step**: Consider running the top Calmar config in paper trading alongside the current live V3 WC NoPool to validate out-of-sample.